In [ ]:
# import libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
sns.set(color_codes=True)
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
import tensorflow as tf

file_name = "run57_mix_mega_shared"
data_dir = "/data/test_newrepo"
model_name = "run53_mix_mega_shared_cart.keras"
test_number = 100000
normalize = 1
os.environ["CUDA_VISIBLE_DEVICES"]="-1"
number_of_detectors = 6


In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:

file_path = data_dir+'/'+file_name+'_dataset.pkl'

# Load the array from the pickle file
with open(file_path, 'rb') as file:
    loaded_array = pickle.load(file)


In [ ]:
# Filter the dataset by spectral model and flux

loaded_array_test = loaded_array

filter_flux =  0 #1:30
filter_spectra = 0

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_flux == 1:
    for grb in loaded_array_test:
        if grb['flux'] > 0 and grb['flux'] <= 10:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_spectra==1:
    for grb in loaded_array_test:
        if  "1500" in grb['spectrum']: #["Band 10 10000 -1.9 -3.7 230","Band 10 10000 -1 -2.3 699.9","Comptonized 10 10000 -0.5 1500"]
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

In [ ]:
loaded_array_test.shape

In [ ]:
filter_len = test_number

random_indices = np.random.permutation(len(loaded_array_test))

shuffled_loaded_array_bkg = loaded_array_test[random_indices]

test_dataset_raw = shuffled_loaded_array_bkg[:filter_len]                               

In [ ]:
test_dataset_raw.shape

In [ ]:

def l2_normalize(data):
  """
  Normalize a NumPy array using the L2 (Euclidean) norm.

  Args:
    data (numpy.ndarray): The array to normalize. Can be a 1D vector
                         or a 2D matrix (where each row is a vector to normalize).

  Returns:
    numpy.ndarray: The L2-normalized array.
  """

  data = np.array(data)
  if data.ndim == 1:
    # Case: 1D vector
    norm = np.sqrt(np.sum(data**2))
    if norm == 0:
      return data  # Avoid division by zero if the vector is null
    return data / norm
  elif data.ndim == 2:
    # Case: 2D matrix (normalize each row)
    norms = np.sqrt(np.sum(data**2, axis=1, keepdims=True))
    # Handle the case of null norms (zero rows)
    norms[norms == 0] = 1
    return data / norms
  else:
    raise ValueError("Input must be a 1D or 2D array.")

In [ ]:
dataset = np.empty((len(test_dataset_raw),number_of_detectors))
labels = np.empty((len(test_dataset_raw),2))

count = -1
for element in test_dataset_raw:

    count+=1
    # We are considering a Poissonian background. The mean counts are calculated from the DC3 BGO data.
    b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])
    dataset[count] = l2_normalize(np.array(element['counts'])+np.random.poisson(b_sim*20)-b_sim*20)
    #dataset[count] = l2_normalize(element['counts'])
    labels[count][0] = float(element['coord'][0])
    labels[count][1] = float(element['coord'][1])
    
labels = labels[:count+1]
dataset = dataset[:count+1]

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Convert to radians
coords_deg = []
for theta, phi in labels:
    coords_deg.append([theta, phi])

In [ ]:
coord_array = np.array(coords_deg)
coord_array.shape

In [ ]:
plt.hist(coord_array[:, 0])
plt.xlabel("Theta")
plt.ylabel("Counts")

In [ ]:
plt.hist(coord_array[:, 1])
plt.xlabel("Phi")
plt.ylabel("Counts")

In [ ]:
def cartesian_to_spherical(x, y, z):
    # Calculate theta
    theta = np.degrees(np.arccos(z))
    
    # Calculate phi
    if x == 0 and y == 0:
        phi = 0
    else:
        phi = np.degrees(np.arctan2(y, x))
        if phi < 0:
            phi += 360
    
    return theta, phi

def process_coordinates(coords_deg):

    # Separate the longitudes and latitudes in radians
    theta, phi = zip(*coords_deg)

    # Convert to Cartesian coordinates
    x = np.sin(np.radians(theta)) * np.cos(np.radians(phi))
    y = np.sin(np.radians(theta)) * np.sin(np.radians(phi))
    z = np.cos(np.radians(theta))
    
    return x,y,z

def process_coordinates_single(coords_deg):

    # Separate the longitudes and latitudes in radians
    theta, phi = coords_deg

    # Convert to Cartesian coordinates
    x = np.sin(np.radians(theta)) * np.cos(np.radians(phi))
    y = np.sin(np.radians(theta)) * np.sin(np.radians(phi))
    z = np.cos(np.radians(theta))
    
    return x,y,z

x,y,z = process_coordinates(coords_deg)


In [ ]:
cartesian_labels = []

for i in range(0,len(test_dataset_raw)):
    cartesian_labels.append([x[i],y[i],z[i]])
    

In [ ]:
theta, phi = zip(*coords_deg)

In [ ]:
def get_radians(coords):

    # Separate the longitudes and latitudes in radians
    theta, phi = zip(*coords)
    
    theta = np.array(theta)
    phi = np.array(phi)
    
   
    mask = phi > 180
    phi[mask] -= 360
    theta = 90-theta

    # Convert to Cartesian coordinates
    return np.radians(theta),np.radians(phi)


In [ ]:
# create cartesian labels dataset
labels_norm = np.empty((len(test_dataset_raw),3))
count = -1
for element in cartesian_labels:
    count+=1
    labels_norm[count][0] = element[0]
    labels_norm[count][1] = element[1]
    labels_norm[count][2] = element[2]

In [ ]:
labels_norm.shape

In [ ]:
plt.hist(labels_norm[:,0])

In [ ]:
plt.hist(labels_norm[:,1])

In [ ]:
plt.hist(labels_norm[:,2])

In [ ]:
test_dataset = dataset
test_labels = labels_norm

In [ ]:
def custom_loss(y_true, y_pred):
    
    lambda_reg = 0.5

    mse_loss = MeanAbsoluteError()(y_true,y_pred)
    
    x_pred, y_pred, z_pred = tf.unstack(y_pred, axis=1)
    sphere_constraint_error = tf.square(x_pred**2 + y_pred**2 + z_pred**2 - 1)
    
    loss = mse_loss**2 + lambda_reg * sphere_constraint_error
    
    return loss

In [ ]:
from tensorflow.keras.models import load_model


# Load the model back from the saved directory
model = load_model(data_dir+"/"+model_name,custom_objects={'custom_loss': custom_loss})

In [ ]:
test_dataset.shape

In [ ]:
test_dataset[0]

In [ ]:

pred_data = model.predict(test_dataset)
    

In [ ]:
pred_data.shape

In [ ]:
test_labels.shape

In [ ]:
import math

pred_data_renorm = np.empty((len(pred_data),3))
test_data_renorm = np.empty((len(test_labels),3))

    
for j in range(0,len(pred_data)):
    pred_data_renorm[j][0]  = pred_data[j][0]
    pred_data_renorm[j][1] = pred_data[j][1]
    pred_data_renorm[j][2] = pred_data[j][2]

    
min_val = 0
max_val = 1

for j in range(0,len(test_labels)):
    test_data_renorm[j][0]  = test_labels[j][0]
    test_data_renorm[j][1] = test_labels[j][1]
    test_data_renorm[j][2] = test_labels[j][2]

      
pred_data_original = np.empty((len(pred_data),2))
test_labels_original = np.empty((len(test_labels),2))
        
for i, element in enumerate(test_data_renorm):

    theta,phi = cartesian_to_spherical(element[0], element[1], element[2])
    
    test_labels_original[i][0]=theta
    test_labels_original[i][1]=phi
    

for i, element in enumerate(pred_data_renorm):

    theta,phi = cartesian_to_spherical(element[0], element[1], element[2])
    
    pred_data_original[i][0]=theta
    pred_data_original[i][1]=phi
        


absolute_diffs_element1 = np.abs(pred_data_original[:, 0] - test_labels_original[:, 0])
absolute_diffs_element2 = np.abs(pred_data_original[:, 1] - test_labels_original[:, 1])

# Calculate the Mean Absolute Error (MAE) for each element separately
mae_element1 = np.mean(absolute_diffs_element1)
mae_element2 = np.mean(absolute_diffs_element2)

print("Mean Absolute Error for Element 1:", mae_element1)
print("Mean Absolute Error for Element 2:", mae_element2)

In [ ]:
plt.hist((pred_data[:,0]))

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels_original[:, 0],bins=50,alpha=0.6,color="b",label="reco coords")
plt.hist(pred_data_original[:,0],bins=50,alpha=0.6,color="r",label="test coords")
plt.xlabel("Theta")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels_original[:, 1],bins=50,alpha=0.6,color="b",label="reco coords")
plt.hist(pred_data_original[:,1],bins=50,alpha=0.6,color="r",label="test coords")
plt.xlabel("Phi")
plt.legend()
plt.ylabel("Counts")

In [ ]:
plt.hist(pred_data_original[:,0])

In [ ]:
pred_data_original

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,0],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 0],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("x")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,1],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 1],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("y")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,2],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 2],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("z")
plt.legend()
plt.ylabel("Counts")


In [ ]:
def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg


In [ ]:
# Calculate distances between corresponding coordinates
#distances = np.linalg.norm(pred_data_original - test_labels_original, axis=1)

distances = []
theta_dist_arr = []
phi_dist_arr = []

#print(pred_data_original)
#print(test_labels_original)

for i in range(0,len(pred_data_original)):

    d = angular_distance(pred_data_original[i][0],pred_data_original[i][1],test_labels_original[i][0],test_labels_original[i][1])
    distances.append(d)
    theta_dist = np.abs(pred_data_original[i][0]-test_labels_original[i][0])
    phi_dist = np.abs(pred_data_original[i][1]-test_labels_original[i][1])
    theta_dist_arr.append(theta_dist)
    phi_dist_arr.append(phi_dist)
    


In [ ]:
print(np.mean(distances))
print(np.mean(theta_dist_arr))
print(np.mean(phi_dist_arr))